# 최종 파이프라인 v21 — 유전자 변이로 암 아형 26종 분류 (Public LB 0.49598)

**구성**: v4s .10 / v4sn .45 / v2(v4p) .20 / v4sp .15 / NB 파트너 .10 → 로그 가중 결합 → 클래스 배율 26개 → argmax → 쌍둥이 규칙.
전부 train 근거. test.csv는 마지막 추론 셀에서만 읽는다. 상세 설명: `2. team/FINAL_REPORT_2026-09-18.md`, 쉬운 버전 `FINAL_REPORT_EZ.md`.

노트북은 저장소 루트에서 실행한다(`4. src/common`을 경로에 추가). 두 경로를 제공한다.
- **A. 빠른 재현(1분)**: 32차 제출 때 저장된 다섯 파트의 test 확률을 그대로 결합 → 제출 파일과 md5까지 동일.
- **B. 전체 학습(약 15분, M1)**: 다섯 파트를 train 전체로 다시 학습해 같은 파일을 만든다(XGB seed 42, 결정적).

In [ ]:
import sys, json, hashlib, inspect, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "4. src").exists() else Path.cwd().parents[0]
sys.path.insert(0, str(ROOT / "4. src/common"))
from main import ROOT, DATA, ID, TARGET, SEED, PARAM_SETS, FeatureMaker, load_train, load_test
from features.spectrum import spectrum_features
from features.spectrum_nb import SpectrumNBFeatures, spectrum_counts
from approach14_cw_plus.blend import run_multi, model_proba, log_blend
from postprocess.twin_rule import TwinRule
print("ROOT =", ROOT)

## 1. 데이터
train 6,201행 × (ID, SUBCLASS, 유전자 4,384). 
값은 `WT` 또는 아미노산 변이 문자열(공백 구분 다중).
'WT' = normal, amino acid 변이 = 암 유발 인자. 
아미노산 변이에 따른 암종 특이성 확인.

In [ ]:
train = load_train()
genes = [c for c in train.columns if c not in (ID, TARGET)]
n_mut = (train[genes] != "WT").sum(axis=1)
print(train.shape, "| 클래스", train[TARGET].nunique())
display(train[TARGET].value_counts().to_frame("n").T)
print("변이 수 분위:", n_mut.quantile([.25, .5, .75, .99]).to_dict())
print("셀 예:", train.loc[0, genes][train.loc[0, genes] != "WT"].head(3).to_dict())

## 2. 피처 — 다섯 파트가 보는 단서
| 파트 | 피처 종류 | 내용 |
|---|---|---|
| v4s | `v4s` | v4 골격(유전자 켜짐/꺼짐 + 변이 개수 + 접근1 클래스별 점수 140 + 인사이트·BLOSUM62 118) + **치환 스펙트럼 411** |
| v4sn | `v4sn` | v4s + **NB 점수표 28열**(치환 프로필 MultinomialNB의 클래스별 로그확률, 내부 5-fold OOF) |
| v2 | `v4p` | v4 + 접근14 **변이 유형 분리 점수** 40열 |
| v4sp | `v4sp` | v4p + 치환 스펙트럼 |
| NB | 383개 횟수 | 치환 프로필 MultinomialNB(α0.5) 파트너 |

아래 셀은 핵심 함수의 소스를 그대로 보여준다.

In [ ]:
print(inspect.getsource(spectrum_features))

In [ ]:
print(inspect.getsource(SpectrumNBFeatures))

In [ ]:
# 피처 열 수 확인 (train 400행 샘플)
sample = train.iloc[:400]
for kind in ["v4", "v4s", "v4sn", "v4p", "v4sp"]:
    X = FeatureMaker(kind).fit(sample).transform(sample)
    print(f"{kind:5s} 열 {X.shape[1]:5d} | cw_ {sum(c.startswith('cw_') for c in X.columns)} | cwp_ {sum(c.startswith('cwp_') for c in X.columns)} | sp {sum(c.startswith('sp') for c in X.columns)} | snb {sum(c.startswith('snb_') for c in X.columns)}")

## 3. 모델과 결합
- 각 XGB 파트: `PARAM_SETS["mild_col"]` = 100그루, 학습률 0.1, 깊이 6, colsample_bytree 0.7, seed 42.
- 결합: 확률의 로그를 가중합 후 softmax(기하평균). 배율: 3차 train OOF로 맞춘 26개 값(고정). 쌍둥이 규칙: train과 완전히 같은 유전자를 가지는 값들은 쌍둥이로 취급해 짝 라벨.

In [ ]:
print(PARAM_SETS["mild_col"])
scales = json.load(open(ROOT / "6. experiments/2026-09-09_v4_xgb_cs/class_scales_recovered.json"))
display(pd.Series(scales).sort_values(ascending=False).to_frame("배율").T)
print(inspect.getsource(log_blend)); print(inspect.getsource(TwinRule.apply))

## 4-A. 빠른 재현 — 저장된 파트 확률로 32차 파일 만들기

In [ ]:
D = ROOT / "6. experiments/submissions/approach14_v19_20260916_1110"   # v4s(part0), v4sn(part1), v4sp(part3), NB(part4) 저장본; v2는 "p2"
part = lambda k: (lambda tr, te: np.load(D / f"test_proba_part{k}.npy"))
NAME = "approach14_v21_notebook"
run_multi(NAME, [(part(0), 0.10), (part(1), 0.45), ("p2", 0.20), (part(3), 0.15), (part(4), 0.10)])
out = ROOT / "6. experiments/submissions/twin_rule" / f"{NAME}_twin_rule.csv"
ref = ROOT / "6. experiments/submissions/twin_rule/approach14_v21_20260917_0900_twin_rule.csv"
print("md5 노트북:", hashlib.md5(out.read_bytes()).hexdigest()); print("md5 32차  :", hashlib.md5(ref.read_bytes()).hexdigest())

## 4-B. 전체 학습 — 다섯 파트를 처음부터 (약 15분)
`model_proba(kind)`는 train 전체로 FeatureMaker(kind)+XGB를 학습해 test 확률을 돌려준다. NB 파트너는 아래 함수.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
def nb_partner(tr, te):
    g = [c for c in tr.columns if c not in (ID, TARGET)]; y = LabelEncoder().fit_transform(tr[TARGET])
    return MultinomialNB(alpha=0.5).fit(spectrum_counts(tr[g].to_numpy()), y).predict_proba(spectrum_counts(te[g].to_numpy()))
RUN_FULL = False   # True로 바꾸면 전체 학습
if RUN_FULL:
    run_multi("approach14_v21_full", [(model_proba("v4s"), 0.10), (model_proba("v4sn"), 0.45), (model_proba("v4p"), 0.20), (model_proba("v4sp"), 0.15), (nb_partner, 0.10)])

## 5. K-fold로 보는 각 파트와 결합 (저장 OOF 재사용)
쌍둥이를 같은 fold에 묶은 5-fold OOF 확률이 `6. experiments/`에 저장돼 있다. 배율 적용 Macro F1.
이를 통해서 교차검증 진행. 

In [ ]:
from sklearn.metrics import f1_score
import glob
le = LabelEncoder().fit(train[TARGET]); y = le.transform(train[TARGET]); C = list(le.classes_)
s3 = np.array([scales[c] for c in C]); E = ROOT / "6. experiments"
L = lambda pat: np.load(sorted(glob.glob(str(E / pat)))[-1] + "/oof_proba.npy")
oof = {"v4s": L("2026-09-14_v4s_xgb_mild_col_grp"), "v4sn": L("2026-09-16_v4sn_xgb_mild_col_grp"), "v2": L("2026-09-13_v4p_xgb_mild_col_grp"), "v4sp": L("*_v4sp_xgb_mild_col_grp"), "NB": np.load(E / "2026-09-15_approach16_spectrum_profile_score/oof_spectrum_profile_score.npy")}
ev = lambda p: round(f1_score(y, (p * s3).argmax(1), average="macro"), 4)
for k, o in oof.items(): print(f"{k:5s} 단독 {ev(o)}")
w = {"v4s": .10, "v4sn": .45, "v2": .20, "v4sp": .15, "NB": .10}
z = sum(np.log(oof[k] + 1e-9) * w[k] for k in w); p = np.exp(z - z.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
print("v21 결합 정직 CV:", ev(p), "| LB 0.49598")

## 6. 규정 준수 체크
- test는 `load_test`(추론 셀)에서만 읽는다. 결측 상수 처리. 인코더·열 목록·점수표·배율·규칙·가중치 전부 train.
- 스펙트럼은 행 내부 비율(fit 없음). NB 점수표 피처는 fold 학습부로만 fit.
- 제출 파일 점검은 형식만(`run_multi`의 assert). 같은 스크립트 두 번 실행 시 md5 동일.
- test DATA는 검증 이외에 사용하지 않는다. 